In [3]:
import torch
import time
import numpy as np
from tqdm import tqdm
import psutil
import subprocess

class RTX5090_Ultimate_Test:
    def __init__(self):
        self.device = torch.device('cuda')
        self.gpu_name = torch.cuda.get_device_name(0)
        self.total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9  # GB
        
        # 设置cuDNN benchmark以获得最佳性能
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
        
        print(f"🚀 RTX 5090 终极性能测试")
        print(f"显卡: {self.gpu_name}")
        print(f"显存: {self.total_memory:.1f} GB")
        print("=" * 70)
    
    def get_gpu_power(self):
        """获取当前GPU功耗"""
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
                capture_output=True, text=True
            )
            return float(result.stdout.strip())
        except:
            return 0.0
    
    def get_gpu_utilization(self):
        """获取GPU利用率"""
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=utilization.gpu', '--format=csv,noheader,nounits'],
                capture_output=True, text=True
            )
            return float(result.stdout.strip())
        except:
            return 0.0
    
    def find_optimal_matrix_size(self):
        """找到能最大利用显存的最优矩阵大小"""
        # 估算可用显存（保留20%给系统）
        available_memory = self.total_memory * 0.7 * 1e9
        
        # 矩阵乘法需要两个输入矩阵和一个输出矩阵
        # 每个FP32元素占4字节
        # 总内存 = 3 * size^2 * 4
        max_size = int(np.sqrt(available_memory / (3 * 4)))
        
        # 调整为64的倍数以获得最佳性能
        optimal_size = (max_size // 64) * 64
        
        # 限制最大尺寸避免OOM
        optimal_size = min(optimal_size, 16384)
        
        print(f"建议矩阵大小: {optimal_size}x{optimal_size}")
        return optimal_size
    
    def test_matmul_batch(self, size, batch_size=8, duration=120):
        """
        使用批处理矩阵乘法进行长时间测试
        duration: 测试持续时间（秒）
        """
        print(f"\n📊 批处理矩阵乘法测试")
        print(f"矩阵大小: {size}x{size}")
        print(f"批大小: {batch_size}")
        print(f"目标持续时间: {duration} 秒")
        print("-" * 70)
        
        # 创建更大的batch
        a = torch.randn(batch_size, size, size, dtype=torch.float32, device=self.device)
        b = torch.randn(batch_size, size, size, dtype=torch.float32, device=self.device)
        
        # 显存使用情况
        memory_used = (batch_size * size * size * 4 * 2) / 1e9  # 两个矩阵
        print(f"显存占用: {memory_used:.1f} GB")
        
        # 预热（让GPU达到稳定状态）
        print("🔥 预热中...")
        for _ in tqdm(range(50), desc="预热进度"):
            c = torch.bmm(a, b)
        torch.cuda.synchronize()
        
        # 正式测试
        print(f"⏱️  开始 {duration} 秒测试...")
        start_time = time.time()
        iterations = 0
        total_flops = 0
        power_readings = []
        util_readings = []
        
        # 创建进度条
        pbar = tqdm(total=duration, desc="测试进度", unit="s")
        
        while time.time() - start_time < duration:
            iter_start = time.time()
            
            # 执行批处理矩阵乘法
            c = torch.bmm(a, b)
            torch.cuda.synchronize()
            
            iter_time = time.time() - iter_start
            iterations += 1
            
            # 计算FLOPs: batch_size * (2 * n^3)
            flops_per_iter = batch_size * 2 * size * size * size
            total_flops += flops_per_iter
            
            # 采集GPU状态（每10次迭代采集一次）
            if iterations % 10 == 0:
                power = self.get_gpu_power()
                util = self.get_gpu_utilization()
                power_readings.append(power)
                util_readings.append(util)
            
            # 更新进度
            elapsed = time.time() - start_time
            pbar.update(elapsed - pbar.n)
            pbar.set_postfix({
                'iter': iterations,
                '功耗': f'{power_readings[-1] if power_readings else 0:.0f}W',
                '利用率': f'{util_readings[-1] if util_readings else 0:.0f}%'
            })
        
        pbar.close()
        elapsed_time = time.time() - start_time
        
        # 计算性能
        avg_tflops = (total_flops / elapsed_time) / 1e12
        avg_power = np.mean(power_readings) if power_readings else 0
        avg_util = np.mean(util_readings) if util_readings else 0
        max_power = max(power_readings) if power_readings else 0
        max_util = max(util_readings) if util_readings else 0
        
        print("\n" + "=" * 70)
        print("📈 测试结果:")
        print(f"总迭代次数: {iterations}")
        print(f"总耗时: {elapsed_time:.2f} 秒")
        print(f"平均每次迭代耗时: {elapsed_time/iterations*1000:.2f} ms")
        print(f"平均性能: {avg_tflops:.2f} TFLOPS")
        print(f"平均功耗: {avg_power:.1f} W")
        print(f"峰值功耗: {max_power:.1f} W")
        print(f"平均GPU利用率: {avg_util:.1f}%")
        print(f"峰值GPU利用率: {max_util:.1f}%")
        
        return {
            'tflops': avg_tflops,
            'iterations': iterations,
            'elapsed': elapsed_time,
            'avg_power': avg_power,
            'max_power': max_power,
            'avg_util': avg_util,
            'max_util': max_util
        }
    
    def test_multi_stream(self, size, duration=120):
        """多流并行测试，充分压榨GPU"""
        print(f"\n⚡ 多流并行测试")
        print(f"矩阵大小: {size}x{size}")
        print(f"流数量: 4")
        print("-" * 70)
        
        # 创建4个CUDA流
        streams = [torch.cuda.Stream() for _ in range(4)]
        
        # 为每个流创建数据
        data = []
        for i in range(4):
            a = torch.randn(size, size, dtype=torch.float32, device=self.device)
            b = torch.randn(size, size, dtype=torch.float32, device=self.device)
            data.append((a, b))
        
        # 预热
        for stream, (a, b) in zip(streams, data):
            with torch.cuda.stream(stream):
                for _ in range(20):
                    c = torch.mm(a, b)
        torch.cuda.synchronize()
        
        # 正式测试
        print(f"⏱️  开始 {duration} 秒测试...")
        start_time = time.time()
        iterations = [0] * 4
        total_flops = 0
        
        pbar = tqdm(total=duration, desc="测试进度", unit="s")
        
        while time.time() - start_time < duration:
            for i, (stream, (a, b)) in enumerate(zip(streams, data)):
                with torch.cuda.stream(stream):
                    c = torch.mm(a, b)
                    iterations[i] += 1
                    total_flops += 2 * size * size * size
            
            # 每100次迭代同步一次
            if sum(iterations) % 100 == 0:
                torch.cuda.synchronize()
            
            elapsed = time.time() - start_time
            pbar.update(elapsed - pbar.n)
            tflops_current = (total_flops / elapsed) / 1e12
            pbar.set_postfix({'TFLOPS': f'{tflops_current:.2f}'})
        
        pbar.close()
        elapsed_time = time.time() - start_time
        avg_tflops = (total_flops / elapsed_time) / 1e12
        
        print("\n" + "=" * 70)
        print(f"📈 多流测试结果:")
        print(f"总迭代次数: {sum(iterations)}")
        print(f"总耗时: {elapsed_time:.2f} 秒")
        print(f"平均性能: {avg_tflops:.2f} TFLOPS")
        
        return avg_tflops
    
    def run_full_test(self):
        """运行完整测试套件"""
        # 自动选择最优矩阵大小
        optimal_size = self.find_optimal_matrix_size()
        
        # 测试1: 批处理矩阵乘法（主要测试）
        result1 = self.test_matmul_batch(
            size=min(optimal_size, 8192),  # 限制大小以保持稳定性
            batch_size=4,
            duration=120
        )
        
        # 测试2: 更大的批处理
        result2 = self.test_matmul_batch(
            size=min(optimal_size, 4096),
            batch_size=16,
            duration=60
        )
        
        # 测试3: 多流测试
        result3 = self.test_multi_stream(
            size=min(optimal_size, 8192),
            duration=60
        )
        
        # 总结
        print("\n" + "=" * 70)
        print("🏆 完整测试总结")
        print("=" * 70)
        print(f"测试1 (批处理4x{optimal_size}x{optimal_size}): {result1['tflops']:.2f} TFLOPS, 功耗 {result1['avg_power']:.0f}W")
        print(f"测试2 (批处理16x4096x4096): {result2['tflops']:.2f} TFLOPS")
        print(f"测试3 (4流并行): {result3:.2f} TFLOPS")
        
        best_tflops = max(result1['tflops'], result2['tflops'], result3)
        print(f"\n🎯 最佳性能: {best_tflops:.2f} TFLOPS")
        print(f"📊 理论峰值利用率: {best_tflops/104.8*100:.1f}%")

if __name__ == "__main__":
    # 安装依赖
    # pip install tqdm psutil
    
    test = RTX5090_Ultimate_Test()
    test.run_full_test()

🚀 RTX 5090 终极性能测试
显卡: NVIDIA GeForce RTX 4070 Laptop GPU
显存: 8.6 GB
建议矩阵大小: 16384x16384

📊 批处理矩阵乘法测试
矩阵大小: 8192x8192
批大小: 4
目标持续时间: 120 秒
----------------------------------------------------------------------
显存占用: 2.1 GB
🔥 预热中...


预热进度: 100%|██████████| 50/50 [00:00<00:00, 411.30it/s]


⏱️  开始 120 秒测试...


测试进度: 120.60468864440918s [02:00,  1.00s/s, iter=240, 功耗=98W, 利用率=100%]                           



📈 测试结果:
总迭代次数: 240
总耗时: 120.60 秒
平均每次迭代耗时: 502.52 ms
平均性能: 8.75 TFLOPS
平均功耗: 107.1 W
峰值功耗: 113.4 W
平均GPU利用率: 98.3%
峰值GPU利用率: 100.0%

📊 批处理矩阵乘法测试
矩阵大小: 4096x4096
批大小: 16
目标持续时间: 60 秒
----------------------------------------------------------------------
显存占用: 2.1 GB
🔥 预热中...


...

⏱️  开始 60 秒测试...


测试进度: 100%|█████████▉| 59.86075282096863/60 [00:59<00:00,  1.00s/s, iter=217, 功耗=89W, 利用率=0%]   c:\Users\cheny\anaconda3\envs\pytorchgpu\lib\site-packages\tqdm\std.py:636: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
测试进度: 100%|██████████| 60.11278057098389/60 [01:00<00:00,  1.00s/s, iter=218, 功耗=89W, 利用率=0%]



📈 测试结果:
总迭代次数: 218
总耗时: 60.15 秒
平均每次迭代耗时: 275.90 ms
平均性能: 7.97 TFLOPS
平均功耗: 92.0 W
峰值功耗: 102.0 W
平均GPU利用率: 63.1%
峰值GPU利用率: 100.0%

⚡ 多流并行测试
矩阵大小: 8192x8192
流数量: 4
----------------------------------------------------------------------
⏱️  开始 60 秒测试...


测试进度: 63.54864048957825s [01:03,  1.00s/s, TFLOPS=10.38]                             


📈 多流测试结果:
总迭代次数: 600
总耗时: 63.58 秒
平均性能: 10.38 TFLOPS

🏆 完整测试总结
测试1 (批处理4x16384x16384): 8.75 TFLOPS, 功耗 107W
测试2 (批处理16x4096x4096): 7.97 TFLOPS
测试3 (4流并行): 10.38 TFLOPS

🎯 最佳性能: 10.38 TFLOPS
📊 理论峰值利用率: 9.9%
